In [1]:
# Notebook setup: import and (optionally) auto-reload local module
import importlib
import time

import locator_time

# If you edit locator_time.py while this notebook is open, re-run this cell to reload.
importlib.reload(locator_time)

print(locator_time.__doc__[:400])

A small, self-contained Python port of ImPlot's time-axis tick locator.

This is based on the logic in `implot.cpp` (function `Locator_Time`) and the
associated helpers in the "Time Ticks and Utils" section.

Goal
----
Given:
  - `t_min` and `t_max` as Unix timestamps in seconds (float or int)
  - `pixels` as the axis pixel length (float or int)

Produce:
  - tick positions (float seconds)
  - lab


## Test out the Data Structures

are these constants used in the #sym:DateTimeSpec date_fmt and time_fmt

Yes.

In your Python port, `DateTimeSpec.date_fmt` is intended to hold one of the `DATE_*` constants, and `DateTimeSpec.time_fmt` is intended to hold one of the `TIMEFMT_*` constants—those enums are the Python equivalents of ImPlot’s `ImPlotDateFmt` and `ImPlotTimeFmt`.

You can see that directly in how the presets are built: `TIME_FORMAT_LEVEL0`, `TIME_FORMAT_LEVEL1`, and `TIME_FORMAT_LEVEL1_FIRST` construct `DateTimeSpec(...)` objects using values like `DATE_NONE`, `DATE_DAY_MO_YR`, `TIMEFMT_US`, `TIMEFMT_HR_MIN`, etc. Those specs are then passed into `format_datetime`, which switches on `spec.date_fmt` and `spec.time_fmt` to choose the right formatting logic.

One nuance: the `use_24_hour` / `use_iso8601` fields exist on `DateTimeSpec`, but your formatting functions are driven by the runtime arguments passed to `format_datetime(...)` (and ultimately `locator_time(...)`). In other words, `date_fmt`/`time_fmt` come from these constants, but the boolean style toggles are currently treated as external “style overrides” rather than being read from the `DateTimeSpec` instance itself.

In [3]:
# test out ImPlotTime from locator_time
locator_time.ImPlotTime(1,1)

ImPlotTime(S=1, Us=1)

In [4]:
# test out ImPlotTime from locator_time
# dir(locator_time.ImPlotTime)

In [5]:
# test out ImPlotTime from locator_time
locator_time.ImPlotTime(1,1)+locator_time.ImPlotTime(1,1)

ImPlotTime(S=2, Us=2)

In [6]:
# test out ImPlotTime from locator_time
(locator_time.ImPlotTime(3,1)+locator_time.ImPlotTime(1,20000)).to_double()

4.020001

In [7]:
locator_time.ImPlotTime(1,20000).roll_over()

ImPlotTime(S=1, Us=20000)

In [8]:
s=locator_time.ImPlotTime(1,20000)

In [9]:
st=locator_time.ImPlotTime(1,2000000)
st

ImPlotTime(S=3, Us=0)

## Test out the helper functions

a quick description of a counter collection

In [6]:
# Small helpers to inspect returned ticks
from collections import Counter

In [7]:
words = ["a", "b", "a", "c", "b", "a"]
c = Counter(words)

print(c)          # Counter({'a': 3, 'b': 2, 'c': 1})
print(c["a"])     # 3
print(c.most_common(2))  # [('a', 3), ('b', 2)]

Counter({'a': 3, 'b': 2, 'c': 1})
3
[('a', 3), ('b', 2)]


In [8]:
items = [
    (0, True,  True),
    (0, False, True),
    (0, True,  True),
    (1, True,  False),
]
c = Counter(items)

print(c)
# Counter({
#   (0, True, True): 2,
#   (0, False, True): 1,
#   (1, True, False): 1
# })

for key, count in c.items():
    print(key, "->", count)

Counter({(0, True, True): 2, (0, False, True): 1, (1, True, False): 1})
(0, True, True) -> 2
(0, False, True) -> 1
(1, True, False) -> 1


In [3]:
def summarize_ticks(ticks):
    c = Counter((t.level, t.major, t.show_label) for t in ticks)
    total = len(ticks)
    level0 = sum(1 for t in ticks if t.level == 0)
    level1 = sum(1 for t in ticks if t.level == 1)
    shown = sum(1 for t in ticks if t.show_label)
    return {
        'total_ticks': total,
        'level0_ticks': level0,
        'level1_ticks': level1,
        'labels_shown': shown,
        'breakdown': dict(c),
    }

def head_ticks(ticks, n=20):
    rows = []
    for t in ticks[:n]:
        tag = f"L{t.level} {'M' if t.major else 'm'}"
        label = t.label if t.show_label else ''
        rows.append((tag, t.pos, label))
    return rows

## Baseline: 1 hour range
This should produce minute/second-ish ticks depending on `pixels` and `max_density`.

In [4]:
now = time.time()
t_min = now
t_max = now + 3600

ticks = locator_time.locator_time(
    t_min, t_max, pixels=800,
    use_local_time=True,
    max_density=0.5,
    char_px=7.0,
)

summarize_ticks(ticks), head_ticks(ticks, 25)

NameError: name 'time' is not defined

In [6]:
ticks

[Tick(pos=1766934000.0, level=0, major=True, show_label=True, label='10:00am'),
 Tick(pos=1766934000.0, level=1, major=True, show_label=True, label='12/28/25'),
 Tick(pos=1766934600.0, level=0, major=False, show_label=True, label='10:10am'),
 Tick(pos=1766935200.0, level=0, major=False, show_label=True, label='10:20am'),
 Tick(pos=1766935800.0, level=0, major=False, show_label=True, label='10:30am'),
 Tick(pos=1766936400.0, level=0, major=False, show_label=True, label='10:40am'),
 Tick(pos=1766937000.0, level=0, major=False, show_label=True, label='10:50am')]

## Compare pixel widths
Smaller `pixels` should suppress more labels (especially level 0 minor labels).

In [8]:
for px in (200, 400, 800, 1200):
    ticks_px = locator_time.locator_time(t_min, t_max, pixels=px, use_local_time=True)
    s = summarize_ticks(ticks_px)
    print(f"pixels={px:4d}  total={s['total_ticks']:4d}  shown={s['labels_shown']:4d}  L0={s['level0_ticks']:4d}  L1={s['level1_ticks']:4d}")

pixels= 200  total=   3  shown=   3  L0=   2  L1=   1
pixels= 400  total=   5  shown=   5  L0=   4  L1=   1
pixels= 800  total=   7  shown=   7  L0=   6  L1=   1
pixels=1200  total=  13  shown=  13  L0=  12  L1=   1


In [11]:
for px in (200, 400, 800, 1200):
    ticks_px = locator_time.locator_time(t_min, t_max, pixels=px, use_local_time=False)
    print(head_ticks(ticks_px))

[('L0 M', 1766934000.0, '3:00pm'), ('L1 M', 1766934000.0, '12/28/25'), ('L0 m', 1766935800.0, '3:30pm')]
[('L0 M', 1766934000.0, '3:00pm'), ('L1 M', 1766934000.0, '12/28/25'), ('L0 m', 1766934900.0, '3:15pm'), ('L0 m', 1766935800.0, '3:30pm'), ('L0 m', 1766936700.0, '3:45pm')]
[('L0 M', 1766934000.0, '3:00pm'), ('L1 M', 1766934000.0, '12/28/25'), ('L0 m', 1766934600.0, '3:10pm'), ('L0 m', 1766935200.0, '3:20pm'), ('L0 m', 1766935800.0, '3:30pm'), ('L0 m', 1766936400.0, '3:40pm'), ('L0 m', 1766937000.0, '3:50pm')]
[('L0 M', 1766934000.0, '3:00pm'), ('L1 M', 1766934000.0, '12/28/25'), ('L0 m', 1766934300.0, '3:05pm'), ('L0 m', 1766934600.0, '3:10pm'), ('L0 m', 1766934900.0, '3:15pm'), ('L0 m', 1766935200.0, '3:20pm'), ('L0 m', 1766935500.0, '3:25pm'), ('L0 m', 1766935800.0, '3:30pm'), ('L0 m', 1766936100.0, '3:35pm'), ('L0 m', 1766936400.0, '3:40pm'), ('L0 m', 1766936700.0, '3:45pm'), ('L0 m', 1766937000.0, '3:50pm'), ('L0 m', 1766937300.0, '3:55pm')]


In [12]:
now = time.time()
t_min = now
t_max = now + 1800

In [13]:
locator_time.locator_time(t_min, t_max, pixels=800, use_local_time=False)

[Tick(pos=1766950200.0, level=0, major=False, show_label=True, label='7:30pm'),
 Tick(pos=1766950200.0, level=1, major=True, show_label=True, label='12/28/25'),
 Tick(pos=1766950500.0, level=0, major=False, show_label=True, label='7:35pm'),
 Tick(pos=1766950800.0, level=0, major=False, show_label=True, label='7:40pm'),
 Tick(pos=1766951100.0, level=0, major=False, show_label=True, label='7:45pm'),
 Tick(pos=1766951400.0, level=0, major=False, show_label=True, label='7:50pm'),
 Tick(pos=1766951700.0, level=0, major=False, show_label=True, label='7:55pm')]

## Explore different spans
These cover typical unit transitions (minutes → hours → days → months → years).

In [ ]:
def run_span(span_seconds, pixels=900, title=None):
    t0 = time.time()
    t1 = t0 + span_seconds
    ticks = locator_time.locator_time(t0, t1, pixels=pixels, use_local_time=True)
    s = summarize_ticks(ticks)
    title = title or f"span={span_seconds}s"
    print(f"\n{title} (pixels={pixels})")
    print(f"  total={s['total_ticks']}  shown={s['labels_shown']}")
    print('  first 12:', head_ticks(ticks, 12))

run_span(10, title='10 seconds')
run_span(5 * 60, title='5 minutes')
run_span(6 * 3600, title='6 hours')
run_span(2 * 86400, title='2 days')
run_span(45 * 86400, title='45 days')
run_span(400 * 86400, title='~400 days (year-ish)')
run_span(10 * 365 * 86400, title='~10 years (year locator)')

## ISO-8601 / 24-hour formatting toggles
These flags match the knobs you might want in a UI layer.

In [14]:
t_min = time.time()
t_max = t_min + 3 * 3600

ticks_default = locator_time.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=False, use_iso8601=False)
ticks_iso24 = locator_time.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=True, use_iso8601=True)

print('default:', head_ticks(ticks_default, 10))
print('iso+24:', head_ticks(ticks_iso24, 10))

default: [('L0 m', 1766957400.0, '4:30pm'), ('L1 M', 1766957400.0, '12/28/25'), ('L0 M', 1766959200.0, '5:00pm'), ('L1 M', 1766959200.0, ''), ('L0 m', 1766961000.0, '5:30pm'), ('L0 M', 1766962800.0, '6:00pm'), ('L1 M', 1766962800.0, ''), ('L0 m', 1766964600.0, '6:30pm'), ('L0 M', 1766966400.0, '7:00pm'), ('L1 M', 1766966400.0, '')]
iso+24: [('L0 m', 1766957400.0, '16:30'), ('L1 M', 1766957400.0, '2025-12-28'), ('L0 M', 1766959200.0, '17:00'), ('L1 M', 1766959200.0, ''), ('L0 m', 1766961000.0, '17:30'), ('L0 M', 1766962800.0, '18:00'), ('L1 M', 1766962800.0, ''), ('L0 m', 1766964600.0, '18:30'), ('L0 M', 1766966400.0, '19:00'), ('L1 M', 1766966400.0, '')]


## Notes on fidelity vs ImPlot
- The biggest mismatch is **text width measurement**: ImPlot uses actual font metrics; here we use `len(label) * char_px`.
- If you have access to real text measurement in your UI, replace `estimate_label_width_px(...)` with it for more faithful label suppression.